# Agricultural Water Demand and Irrigation Scheduling

This notebook walks through a small FAO-56 workflow for maize: it fetches daily weather inputs when Open-Meteo is available, falls back to a bundled sample CSV when it is not, computes reference ET$_0$, estimates crop water demand, and builds a simple irrigation schedule with a soil-water balance. The goal is to show the whole path from weather data to actionable irrigation numbers in a compact, reproducible tutorial.

## Setup

Install the optional extras used in this notebook:

```bash
pip install "aquascope[viz]"
# or, for the minimal package:
pip install aquascope
```

In [ ]:
from datetime import date
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / "aquascope").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from aquascope.agri import (
    crop_water_requirement,
    fetch_openmeteo_plan_inputs,
    get_kc,
    hargreaves,
    irrigation_schedule,
    penman_monteith_daily,
    plan_irrigation,
    penman_monteith_series,
)
from aquascope.agri.water_balance import SoilProperties

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

## Fetch weather inputs

We try Open-Meteo first. If the request fails or returns empty data, the notebook loads the bundled CSV fallback in `notebooks/data/agri_weather_sample.csv`. The same fallback file is also used below to demonstrate the full weather-to-ET$_0$ calculation locally.

In [ ]:
latitude = 35.0
longitude = 139.0
elevation = 120.0
planting_date = date(2026, 4, 1)
season_start = planting_date
season_end = date(2026, 7, 24)

data_path = repo_root / "notebooks" / "data" / "agri_weather_sample.csv"
sample_weather_df = pd.read_csv(data_path, parse_dates=["date"])
sample_weather_df = sample_weather_df.set_index("date").sort_index()

weather_cols = ["t_min", "t_max", "rh_min", "rh_max", "wind_speed", "solar_radiation"]
pm_weather_df = sample_weather_df[weather_cols].rename(columns={"wind_speed": "wind_speed", "solar_radiation": "solar_radiation"})

sample_pm_eto = penman_monteith_series(pm_weather_df, latitude=latitude, elevation=elevation)
sample_precip = sample_weather_df["precipitation_sum"]

try:
    fetched = fetch_openmeteo_plan_inputs(latitude, longitude, season_start.isoformat(), season_end.isoformat())
    if fetched is None:
        raise ValueError("Open-Meteo returned no data")
    plan_eto_series, plan_precip_series = fetched
    plan_source = "Open-Meteo"
except Exception as exc:
    plan_eto_series = sample_pm_eto.copy()
    plan_precip_series = sample_precip.copy()
    plan_source = f"sample CSV fallback ({exc.__class__.__name__})"

display(pd.DataFrame({"eto_mm": plan_eto_series.head(), "precip_mm": plan_precip_series.head()}))
print(f"Weather source for the planning workflow: {plan_source}")

## Reference ET$_0$

The bundled weather file lets us demonstrate the daily FAO-56 calculation directly. The first call uses `penman_monteith_series` on a weather DataFrame, and the second shows a one-day `penman_monteith_daily` example. A simple `hargreaves` call is included as a temperature-only fallback.

In [ ]:
daily_row = sample_weather_df.iloc[0]
daily_doy = daily_row.name.timetuple().tm_yday

eto_one_day = penman_monteith_daily(
    t_min=float(daily_row["t_min"]),
    t_max=float(daily_row["t_max"]),
    rh_min=float(daily_row["rh_min"]),
    rh_max=float(daily_row["rh_max"]),
    u2=float(daily_row["wind_speed"]),
    rs=float(daily_row["solar_radiation"]),
    latitude=latitude,
    elevation=elevation,
    doy=daily_doy,
)

ra = 24.0  # simple example value for the Hargreaves demonstration
eto_hargreaves = hargreaves(float(daily_row["t_min"]), float(daily_row["t_max"]), ra)

print("First five ET0 values from the sample weather DataFrame:")
display(sample_pm_eto.head().to_frame(name="eto_mm"))
print(f"Single-day Penman-Monteith ET0: {eto_one_day:.2f} mm/day")
print(f"Hargreaves example ET0: {eto_hargreaves:.2f} mm/day")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
sample_pm_eto.plot(ax=ax, color="#0f766e", lw=2)
ax.set_title("Reference ET$_0$ from the sample weather file")
ax.set_ylabel("ET$_0$ (mm/day)")
ax.set_xlabel("Date")
plt.tight_layout()

## Crop coefficients

For this tutorial we use maize. The single crop coefficient starts low during establishment, rises through development, reaches its mid-season plateau, and tapers during late season.

In [ ]:
kc = get_kc("maize")
display(pd.Series(kc, name="Kc"))
print("Initial = establishment, mid = peak canopy demand, late = senescence / harvest decline.")

## Crop water requirement

Next we turn ET$_0$ into crop evapotranspiration for maize and summarize the growing-season totals.

In [ ]:
cwr_df = crop_water_requirement(
    plan_eto_series,
    crop="maize",
    planting_date=planting_date,
)

totals = {
    "total_etc_mm": round(float(cwr_df["etc"].sum()), 2),
    "effective_rain_mm": round(float(cwr_df.get("effective_rain", pd.Series(dtype=float)).sum()), 2),
    "net_irrigation_mm": round(float(cwr_df.get("net_irrigation", pd.Series(dtype=float)).sum()), 2),
}
display(cwr_df[["date", "stage", "kc", "eto", "etc"]].head())
print(totals)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(cwr_df["date"], cwr_df["etc"], color="#b45309", lw=2)
ax.set_title("Maize crop water requirement")
ax.set_ylabel("ETc (mm/day)")
ax.set_xlabel("Date")
plt.tight_layout()

## Irrigation scheduling

The schedule combines ETc with precipitation. We show the simple FAO-56 daily schedule and a full soil-water-balance plan with a default sandy-loam style soil profile.

In [ ]:
sched = irrigation_schedule(
    plan_eto_series,
    plan_precip_series,
    crop="maize",
    planting_date=planting_date,
    efficiency=0.7,
)

soil = SoilProperties(field_capacity=0.30, wilting_point=0.15, root_depth=1.0)
plan = plan_irrigation(
    crop="maize",
    planting_date=planting_date,
    eto_series=plan_eto_series,
    precip_series=plan_precip_series,
    soil=soil,
    efficiency=0.7,
)

display(sched[["date", "etc", "effective_rain", "net_irrigation", "gross_irrigation"]].head())
print(f"Irrigation events: {int((sched["gross_irrigation"] > 0).sum())}")
print(f"Gross irrigation from plan_irrigation: {plan.total_gross_irrigation_mm:.2f} mm")

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(sched["date"], sched["gross_irrigation"], width=0.8, alpha=0.35, label="Gross irrigation")
ax.plot(sched["date"], sched["net_irrigation"], color="#1d4ed8", lw=2, label="Net irrigation")
trigger_days = sched.loc[sched["gross_irrigation"] > 0, "date"]
for d in trigger_days:
    ax.axvline(d, color="#dc2626", alpha=0.08)
ax.set_title("Net vs gross irrigation for maize")
ax.set_ylabel("Irrigation (mm/day)")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()

## Summary

A short table makes the season-level picture easier to read.

In [ ]:
summary = pd.DataFrame(
    [{
        "season_start": season_start,
        "season_end": season_end,
        "total_etc_mm": round(float(sched["etc"].sum()), 2),
        "effective_rain_mm": round(float(sched["effective_rain"].sum()), 2),
        "net_irrigation_mm": round(float(sched["net_irrigation"].sum()), 2),
        "gross_irrigation_mm": round(float(sched["gross_irrigation"].sum()), 2),
        "irrigation_events": int((sched["gross_irrigation"] > 0).sum()),
    }]
)
display(summary)

print(
    "The season starts with low crop water demand, then ETc rises as the maize canopy develops and reaches its peak mid-season. "
    "Rainfall offsets part of the demand, but the drier stretches still require irrigation to avoid depletion beyond the soil's readily available water. "
    "Because gross irrigation is adjusted for 70% efficiency, the pumped volume is higher than the net water that actually reaches the root zone. "
    "The plan_irrigation result adds a soil-water balance layer, which is useful for checking when the root zone crosses the irrigation trigger. "
    "In practice, this notebook gives you a simple starting point for comparing planting dates, crop choices, and irrigation efficiency assumptions."
)

## Error handling note

When a user asks for a crop that is not in the FAO-56 lookup table, AquaScope raises a clear error. Catching that error lets you show a friendlier message in notebooks or apps.

In [ ]:
try:
    get_kc("invalid_crop")
except ValueError as exc:
    print(f"Could not look up crop coefficients: {exc}")